## Statistical Hypothesis Testing

---
## Step 1 - Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

TITLE_TEAMS = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']

TEAM_PALETTE = {
    'Arsenal':            '#EF0107',
    'Liverpool':          '#00B2A9',
    'Manchester City':    '#6CABDD',
    'Manchester United':  '#FFB81C',
}

rng = np.random.default_rng(42)

print('Imports ready')

Imports ready


---
## Step 2 - Load Data

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['team', 'date']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Teams: {sorted(df["team"].unique())}')

Shape: (1064, 37)
Teams: ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']


---
## Section A - Normality Checks

In [3]:
# Shapiro-Wilk on points, xG, and match_drop_index
# decides which tests get a non-parametric companion later
ars = df[df['team'] == 'Arsenal'].copy()
season_xga_avg = ars.groupby('season')['xGA'].transform('mean')
season_xg_avg = ars.groupby('season')['xG'].transform('mean')
ars['match_drop_index'] = ((season_xg_avg - ars['xG']) + (season_xga_avg - ars['xGA'])) / 2

for label, series in [('points (0/1/3, discrete)', ars['points']),
                     ('xG', ars['xG']),
                     ('match_drop_index', ars['match_drop_index'])]:
    stat, p = stats.shapiro(series)
    verdict = 'NOT Normal' if p < 0.05 else 'Consistent with Normal'
    print(f'{label:28s} Shapiro-Wilk p={p:.4f}  ->  {verdict}')

points (0/1/3, discrete)     Shapiro-Wilk p=0.0000  ->  NOT Normal
xG                           Shapiro-Wilk p=0.0000  ->  NOT Normal
match_drop_index             Shapiro-Wilk p=0.0003  ->  NOT Normal


### Reading Section A

---
## Section B - xG/xGA Asymmetry, Corrected

In [4]:
# Welch's t-test and Mann-Whitney U, per team, xG and xGA (8 tests)
ttest_rows=[]
mwu_rows = []

for team in TITLE_TEAMS:
    t = df[df['team'] == team]
    hs = t[t['is_high_stakes_retro']]
    nm = t[~t['is_high_stakes_retro']]

    _, p_xg_t = stats.ttest_ind(hs['xG'], nm['xG'], equal_var = False)
    _, p_xga_t = stats.ttest_ind(hs['xGA'], nm['xGA'], equal_var = False)
    ttest_rows.append({'team' : team, 'xG_p' : p_xg_t, 'xGA_p' : p_xga_t})

    _, p_xg_u = stats.mannwhitneyu(hs['xG'], nm['xG'], alternative = 'two-sided')
    _, p_xga_u = stats.mannwhitneyu(hs['xGA'], nm['xGA'], alternative = 'two-sided')
    mwu_rows.append({'team': team, 'xG_p': p_xg_u, 'xGA_p': p_xga_u})

ttest_df = pd.DataFrame(ttest_rows).set_index('team')
mwu_df = pd.DataFrame(mwu_rows).set_index('team')

print("Welch's t-test (parametric):")
print(ttest_df.round(4))
print()
print('Mann-Whitney U (non-parametric):')
print(mwu_df.round(4))

Welch's t-test (parametric):
                     xG_p   xGA_p
team                             
Arsenal            0.6648  0.0309
Liverpool          0.0805  0.5755
Manchester City    0.8425  0.5746
Manchester United  0.2405  0.0325

Mann-Whitney U (non-parametric):
                     xG_p   xGA_p
team                             
Arsenal            0.6360  0.0205
Liverpool          0.1681  0.9416
Manchester City    0.9401  0.4797
Manchester United  0.1530  0.0256


In [5]:
# Bonferroni and Benjamini-Hochberg correction across the 8 tests

def correct_and_show(pvals_dict, method_name):
    labels = list(pvals_dict.keys())
    pvals = list(pvals_dict.values())
    reject_bonf, p_bonf, _, _ = multipletests(pvals, alpha = 0.05, method='bonferroni')
    reject_bh, p_bh, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
    out = pd.DataFrame({
        'raw_p': pvals,
        'bonferroni_p': p_bonf,
        'bonferroni_sig': reject_bonf,
        'bh_p': p_bh,
        'bh_sig': reject_bh,
    }, index=labels)
    print(f'--- {method_name}: corrected across {len(pvals)} tests ---')
    print(out.round(3))
    print()
    return out

ttest_pvals = {}
mwu_pvals = {}
for team in TITLE_TEAMS:
    ttest_pvals[f'{team} xG'] = ttest_df.loc[team, 'xG_p']
    ttest_pvals[f'{team} xGA'] = ttest_df.loc[team, 'xGA_p']
    mwu_pvals[f'{team} xG'] = mwu_df.loc[team, 'xG_p']
    mwu_pvals[f'{team} xGA'] = mwu_df.loc[team, 'xGA_p']

ttest_corrected = correct_and_show(ttest_pvals, "Welch's t-test")
mwu_corrected = correct_and_show(mwu_pvals, 'Mann-Whitney U')

--- Welch's t-test: corrected across 8 tests ---
                       raw_p  bonferroni_p  bonferroni_sig   bh_p  bh_sig
Arsenal xG             0.665         1.000           False  0.760   False
Arsenal xGA            0.031         0.247           False  0.130   False
Liverpool xG           0.081         0.644           False  0.215   False
Liverpool xGA          0.576         1.000           False  0.760   False
Manchester City xG     0.842         1.000           False  0.842   False
Manchester City xGA    0.575         1.000           False  0.760   False
Manchester United xG   0.241         1.000           False  0.481   False
Manchester United xGA  0.032         0.260           False  0.130   False

--- Mann-Whitney U: corrected across 8 tests ---
                       raw_p  bonferroni_p  bonferroni_sig   bh_p  bh_sig
Arsenal xG             0.636         1.000           False  0.848   False
Arsenal xGA            0.020         0.164           False  0.103   False
Liverpool xG 

### Reading Section B

---
## Section C - Bottle Gap Significance

---
### Match-level permutation test

In [13]:
# permutation test per team-season: resample which 10 of 38 matches are
# "high stakes", 10,000 draws, empirical p-value. Correct across all 28.
toy_points = np.array([3, 1, 0, 3, 1, 3])
toy_hs_idx = [4,5]
observed_gap = toy_points[toy_hs_idx].mean() - toy_points.mean()

print(f'Toy observed gap: {observed_gap:+.3f}')

#Every possible way to choose 2 of the 6 matches as "high stakes"
from itertools import combinations
null_gaps = []
for combo in combinations(range(6), 2):
    gap = toy_points[list(combo)].mean() - toy_points.mean()
    null_gaps.append(gap)

null_gaps = np.array(null_gaps)
p_exact = np.mean(np.abs(null_gaps) >= abs(observed_gap))
print(f'All {len(null_gaps)} possible 2-of-6 splits, gaps: {np.round(null_gaps, 2)}')
print(f'Exact permutation p-value: {p_exact:.3f}')

Toy observed gap: +0.167
All 15 possible 2-of-6 splits, gaps: [ 0.17 -0.33  1.17  0.17  1.17 -1.33  0.17 -0.83  0.17 -0.33 -1.33 -0.33
  0.17  1.17  0.17]
Exact permutation p-value: 1.000


### Reading the permutation results

---
### Season-level Wilcoxon signed-rank (avoids pseudoreplication)

In [17]:
# one PPG gap value per team-season (n=7 per team, n=28 pooled),
# Wilcoxon signed-rank against zero
def permutation_test(team, season, n_perm=10000):
    t = df[(df['team'] == team) & (df['season'] == season)]
    pts = t['points'].values
    n_hs = t['is_high_stakes_retro'].sum()
    observed_gap = t.loc[t['is_high_stakes_retro'], 'points'].mean() - pts.mean()

    idx = np.arange(len(pts))
    null_gaps = np.empty(n_perm)
    for i in range(n_perm):
        sel = rng.choice(idx, size = n_hs, replace = False)
        null_gaps[i] = pts[sel].mean() - pts.mean()

    p = np.mean(np.abs(null_gaps) >= abs(observed_gap))
    return observed_gap, p

SEASONS = [1920,2021,2122,2223,2324,2425,2526]
perm_rows = []
for team in TITLE_TEAMS:
    for season in SEASONS:
        gap, p = permutation_test(team,season)
        perm_rows.append({'team': team, 'season': season, 'gap':gap, 'p_raw':p})

perm_df = pd.DataFrame(perm_rows)
reject_bonf, p_bonf, _, _ = multipletests(perm_df['p_raw'], alpha = 0.5, method = 'bonferroni')
reject_bh, p_bh, _, _ = multipletests(perm_df['p_raw'], alpha=0.05, method='fdr_bh')
perm_df['p_bonf'] = p_bonf
perm_df['p_bh'] = p_bh
perm_df['sig_raw'] = perm_df['p_raw'] < 0.05

print(perm_df.round(3).to_string(index=False))
print()
print(f'Raw p < 0.05: {perm_df["sig_raw"].sum()} / 28')
print(f'Survives Bonferroni: {(p_bonf < 0.05).sum()} / 28')
print(f'Survives BH: {(p_bh < 0.05).sum()} / 28')

             team  season    gap  p_raw  p_bonf  p_bh  sig_raw
          Arsenal    1920  0.226  0.558   1.000 0.810    False
          Arsenal    2021  0.195  0.694   1.000 0.810    False
          Arsenal    2122 -0.616  0.112   1.000 0.707    False
          Arsenal    2223 -0.111  0.741   1.000 0.830    False
          Arsenal    2324  0.158  0.657   1.000 0.810    False
          Arsenal    2425 -0.047  1.000   1.000 1.000    False
          Arsenal    2526  0.163  0.630   1.000 0.810    False
        Liverpool    1920 -0.605  0.028   0.776 0.707     True
        Liverpool    2021  0.184  0.678   1.000 0.810    False
        Liverpool    2122  0.179  0.597   1.000 0.810    False
        Liverpool    2223  0.637  0.085   1.000 0.707    False
        Liverpool    2324 -0.158  0.619   1.000 0.810    False
        Liverpool    2425 -0.511  0.116   1.000 0.707    False
        Liverpool    2526 -0.179  0.690   1.000 0.810    False
  Manchester City    1920 -0.032  1.000   1.000 1.000  

### Reading the Wilcoxon results

---
## Section D - Cross-Team Comparison

In [8]:
# Kruskal-Wallis across the 4 teams' season-level PPG gaps


### Reading Chart D1

---
## Section E - Bootstrap Confidence Intervals

In [9]:
# 95% bootstrap CI (resample with replacement) for each team's
# Pressure Resilience Index


In [10]:
# error-bar chart, CI per team


### Reading Chart E1

---
## Section F - Effect Size and Statistical Power

In [11]:
# minimum detectable Cohen's d at n=10 vs n=28, alpha=0.05, power=0.8
# compare against observed Cohen's d in Arsenal's 4 headline seasons


### Reading Section F

---
## Key Findings Summary